# Cohort B: build the patient-episode feature table

Mirrors `04_set_up_cohort_a_feature_table`, but for Cohort B. Since Cohort B episodes are EHR-only
(no registry confirmation), the match-quality/registry-matching step is replaced with the care-site
filter from `03_cohort_b_care_site_assignment`. Produces
`cohort_b_derived.cohort_b_feature_table`.

In [ ]:
USE CATALOG 
your_catalog;

### Restrict to the 11 CIPOC cancer types, at qualifying care sites
Joins `cohort_b_derived.cohort_b_source` (from `02_set_up_cohort_b_cartesian`) to
`cohort_b_derived.cohort_b_care_sites` (from `03_cohort_b_care_site_assignment`) and restricts to
the 11 CIPOC cancer types.

In [ ]:
--flag registry/ehr matches that don't seem to be good quality
--limit ehr rows to the 11 CIPOC cancers
--automatically drops the reg-only people
drop view if exists filtered_cases;
create view filtered_cases as
select b.*,
'ok' as match_quality
from cohort_b_derived.cohort_b_source b JOIN cohort_b_derived.cohort_b_care_sites c ON b.ehr_person_id = c.ehr_person_id and b.ehr_rollup2 = c.ehr_rollup2 and b.ehr_episode_start = c.index_date
WHERE b.ehr_rollup2 IN ('Leukemia','Colon and rectum','Digestive system','Respiratory system','Lymphoma','Female genital system','Myeloma','Male genital system','Breast','Female genital system','Urinary system')

### Build the 6-month post-diagnosis contact window
Same visit-merging logic as Cohort A: overlapping/adjacent visits in the 180 days after diagnosis
are merged into contact-day "islands," and the total contact days and years of UNC history are
computed per episode.

In [ ]:
drop view if exists full_patient_list_filtered;
create temp view full_patient_list_filtered as

with preagg as (
    select *, ehr_episode_start as period_start, date_add(ehr_episode_start,180) as period_end from filtered_cases),

--this query basically creates mini episodes for each patient out of the visits  
clipped as (
    select 
    p.*,
    greatest(v.visit_start_date, period_start) as admit_date, 
    least(v.visit_end_date, period_end) as discharge_date
    from preagg p 
    left join cohort_b.visit_occurrence v ON p.ehr_person_id = v.person_id
    where v.VISIT_START_DATE <= p.period_end AND v.VISIT_END_DATE >= period_start
),

-- this query creates a boolean var that is basically like a true or false answering the question: does this new visit occur after the end of the previous visit? 1 = true
flagged as (
    select *,
    case when admit_date > max(discharge_date) over (partition by ehr_person_id order by admit_date rows between unbounded preceding and 1 preceding) then 1 else 0 
    end as is_new_visit
    from clipped c
), 
-- this query groups all overlapping visits into islands 
-- and the boolean var is a true or false of this question: is this visit on the previous island or is it a new island 
visit_island as ( 
    select *,
    sum(is_new_visit) over (partition by ehr_person_id order by admit_date) as visit_island_id 
    from flagged
    ),
  --then I add it all up :) and back to the previous code we were using 
preagg2 as(
   select ehr_person_id, ehr_episode_start, ehr_episode_end, ehr_rollup2, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, sum(visit_days) as total_contact_days 
from 
(select ehr_person_id, ehr_episode_start, ehr_episode_end, ehr_rollup2, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, visit_island_id, 
date_diff(max(discharge_date), min((admit_date))) + 1 as visit_days 
from visit_island 
group by ehr_person_id,ehr_episode_start,ehr_episode_end, ehr_rollup2, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, visit_island_id) as per_visit_island
group by ehr_person_id,ehr_episode_start, ehr_episode_end, ehr_rollup2, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality),

preagg3 as (
  select p.ehr_person_id, p.ehr_episode_start, p.ehr_episode_end, p.ehr_rollup2, p.registry_person_id, p.registry_cancer_dx_date, p.registry_site, p.match_type, p.daysbt, p.match_quality, total_contact_days, max(v.visit_start_date) as maxunc, min(v.visit_start_date) as minunc
  from preagg2 p JOIN cohort_b.visit_occurrence v ON p.ehr_person_id = v.person_id
  group by p.ehr_person_id, p.ehr_episode_start, p.ehr_episode_end, p.ehr_rollup2, p.registry_person_id, p.registry_cancer_dx_date, p.registry_site, p.match_type, p.daysbt, p.match_quality, total_contact_days
)

select ehr_person_id, ehr_episode_start, ehr_episode_end, ehr_rollup2 as ehr_rollup2, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, total_contact_days, date_diff(maxunc, minunc)/365.25 as years_w_unc
from preagg3

Adds `avg_days_bt_visits`: average gap between visits in the 6-month post-diagnosis window.

In [ ]:
drop view if exists full_patient_list_filtered_lag;
create temp view full_patient_list_filtered_lag as

--calculate avg time between visits in 6 mo after dx
with one as (
select distinct f.*, v.visit_start_date
from full_patient_list_filtered f JOIN cohort_b.visit_occurrence v ON f.ehr_person_id = v.person_id and  v.visit_start_date between coalesce(f.registry_cancer_dx_date,f.ehr_episode_start) and date_add(coalesce(f.registry_cancer_dx_date,f.ehr_episode_start),180)
),

lagfunction as (
select distinct f.*, visit_start_date,
datediff(visit_start_date,lag(visit_start_date) OVER (PARTITION BY ehr_person_id, ehr_rollup2 ORDER BY visit_start_date)) AS days_bt_visits
from one f 
)

select ehr_person_id, ehr_episode_start, ehr_episode_end, ehr_rollup2, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, total_contact_days, years_w_unc, avg(days_bt_visits) as avg_days_bt_visits
from lagfunction
group by ehr_person_id, ehr_episode_start, ehr_episode_end, ehr_rollup2, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, total_contact_days, years_w_unc

Adds `rad_days`: count of distinct dates with a radiation procedure in the same window.

In [ ]:
--create radiation feature
--get any radiation that happened within six months of episode start
drop view if exists full_patient_list_filtered_rad;
create temp view full_patient_list_filtered_rad as
with preaggproc as (
select distinct f.*, p.procedure_concept_id, p.procedure_date
from cohort_b.procedure_occurrence p JOIN reference.radsurgchemocodes r ON r.standard_concept_id = p.procedure_concept_id and lower(r.concept_type) = 'radiation' RIGHT JOIN full_patient_list_filtered_lag f ON f.ehr_person_id = p.person_id and (p.procedure_date between coalesce(f.registry_cancer_dx_date,f.ehr_episode_start) and date_add(coalesce(f.registry_cancer_dx_date,f.ehr_episode_start),180) or p.procedure_date is null))

select ehr_person_id, ehr_episode_start, ehr_episode_end, ehr_rollup2, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, total_contact_days, years_w_unc, avg_days_bt_visits, count(distinct procedure_date) as rad_days
from preaggproc
group by ehr_person_id, ehr_episode_start, ehr_episode_end, ehr_rollup2, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, total_contact_days, years_w_unc, avg_days_bt_visits

Adds `surg_days`: count of distinct dates with a surgery procedure in the same window.

In [ ]:
--create surgery feature
--get any surgery that happened within six months of episode start
drop view if exists full_patient_list_filtered_rad_surg;
create temp view full_patient_list_filtered_rad_surg as
with preaggproc as (
select distinct f.*, p.procedure_concept_id, p.procedure_date
from cohort_b.procedure_occurrence p JOIN reference.radsurgchemocodes r ON r.standard_concept_id = p.procedure_concept_id and lower(r.concept_type) = 'surgery' RIGHT JOIN full_patient_list_filtered_rad f ON f.ehr_person_id = p.person_id and (p.procedure_date between coalesce(f.registry_cancer_dx_date,f.ehr_episode_start) and date_add(coalesce(f.registry_cancer_dx_date,f.ehr_episode_start),180) or p.procedure_date is null))

select ehr_person_id, ehr_episode_start, ehr_episode_end, ehr_rollup2, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, total_contact_days, years_w_unc, avg_days_bt_visits, rad_days, count(distinct procedure_date) as surg_days
from preaggproc
group by ehr_person_id, ehr_episode_start, ehr_episode_end, ehr_rollup2, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, total_contact_days, years_w_unc, avg_days_bt_visits, rad_days

Adds `chemproc_days`: count of distinct dates with a chemotherapy procedure in the same window.

In [ ]:
--create chemo proc feature
--get any chemo proc that happened within six months of episode start
drop view if exists full_patient_list_filtered_rad_surg_chemproc;
create temp view full_patient_list_filtered_rad_surg_chemproc as
with preaggproc as (
select distinct f.*, p.procedure_concept_id, p.procedure_date
from cohort_b.procedure_occurrence p JOIN reference.radsurgchemocodes r ON r.standard_concept_id = p.procedure_concept_id and lower(r.concept_type) = 'chemo' RIGHT JOIN full_patient_list_filtered_rad_surg f ON f.ehr_person_id = p.person_id and (p.procedure_date between coalesce(f.registry_cancer_dx_date,f.ehr_episode_start) and date_add(coalesce(f.registry_cancer_dx_date,f.ehr_episode_start),180) or p.procedure_date is null))

select ehr_person_id, ehr_episode_start, ehr_episode_end, ehr_rollup2, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, total_contact_days, years_w_unc, avg_days_bt_visits, rad_days, surg_days, count(distinct procedure_date) as chemproc_days
from preaggproc
group by ehr_person_id, ehr_episode_start, ehr_episode_end, ehr_rollup2, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, total_contact_days, years_w_unc, avg_days_bt_visits, rad_days, surg_days

Adds `chemdrug_inst`: count of distinct dates with a chemotherapy drug order in the same window.

In [ ]:
--find chemo drug orders within six months of dx date
drop view if exists full_patient_list_filtered_rad_surg_chemproc_drug;
create temp view full_patient_list_filtered_rad_surg_chemproc_drug as
with preaggproc as (
select distinct f.*, p.drug_concept_id, p.drug_exposure_start_date
from cohort_b.drug_exposure p JOIN reference.radsurgchemocodes r ON r.standard_concept_id = p.drug_concept_id and lower(r.concept_type) = 'chemo' RIGHT JOIN full_patient_list_filtered_rad_surg_chemproc f ON f.ehr_person_id = p.person_id and (p.drug_exposure_start_date between coalesce(f.registry_cancer_dx_date,f.ehr_episode_start) and date_add(coalesce(f.registry_cancer_dx_date,f.ehr_episode_start),180) or p.drug_exposure_start_date is null))

select ehr_person_id, ehr_episode_start, ehr_episode_end, ehr_rollup2, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, total_contact_days, years_w_unc, avg_days_bt_visits, rad_days, surg_days, chemproc_days, count(distinct drug_exposure_start_date) as chemdrug_inst
from preaggproc
group by ehr_person_id, ehr_episode_start, ehr_episode_end, ehr_rollup2, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, total_contact_days, years_w_unc, avg_days_bt_visits, rad_days, surg_days, chemproc_days

Maps EHR condition concept IDs back to ICD-10-CM "C" codes (same approach as earlier notebooks).

In [ ]:
--convert condition_concept_ids that are actually in the data back to ICD-10s
drop view if exists all_c_codes_in_data;
create view all_c_codes_in_data as 
SELECT distinct c.concept_id as original_concept_id, c2.*
FROM cohort_a.concept c JOIN cohort_b.condition_occurrence co ON c.concept_id = co.condition_concept_id 
    JOIN cohort_a.concept_relationship cr ON c.concept_id = cr.concept_id_2 and relationship_id = 'Maps to'
    JOIN cohort_a.concept c2 ON c2.concept_id = cr.concept_id_1 and c2.vocabulary_id = 'ICD10CM' and c2.concept_code LIKE 'C%' and c2.invalid_reason is null

Adds `ccode_days`: count of distinct dates with any cancer ("C"-coded) diagnosis in the same window.

In [ ]:
--get a count of C codes in the six months post-dx
drop view if exists full_patient_list_filtered_rad_surg_chemproc_drug_dx;
create temp view full_patient_list_filtered_rad_surg_chemproc_drug_dx as
with preaggproc as (
select distinct f.*, co.condition_concept_id, co.condition_start_date
from cohort_b.condition_occurrence co JOIN all_c_codes_in_data ac ON co.condition_concept_id = ac.original_concept_id
RIGHT JOIN full_patient_list_filtered_rad_surg_chemproc_drug f ON f.ehr_person_id = co.person_id and (co.condition_start_date between coalesce(f.registry_cancer_dx_date,f.ehr_episode_start) and date_add(coalesce(f.registry_cancer_dx_date,f.ehr_episode_start),180) or co.condition_start_date is null))

select ehr_person_id, ehr_episode_start, ehr_episode_end, ehr_rollup2, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, total_contact_days, years_w_unc, avg_days_bt_visits, rad_days, surg_days, chemproc_days, chemdrug_inst, count(distinct condition_start_date) as ccode_days
from preaggproc
group by ehr_person_id, ehr_episode_start, ehr_episode_end, ehr_rollup2, registry_person_id, registry_cancer_dx_date, registry_site, match_type, daysbt, match_quality, total_contact_days, years_w_unc, avg_days_bt_visits, rad_days, surg_days, chemproc_days, chemdrug_inst

### Final Cohort B feature table
Combines the counts above into treatment-intensity ratios and saves
`cohort_b_derived.cohort_b_feature_table`.

In [ ]:
--create final feature table
drop table if exists cohort_b_derived.cohort_b_feature_table;
create table cohort_b_derived.cohort_b_feature_table as
select distinct ehr_person_id, ehr_episode_start, registry_site as reg_cx_type, ehr_rollup2 as ehr_cx_type, match_type, total_contact_days, years_w_unc, avg_days_bt_visits, rad_days, surg_days, chemproc_days, chemdrug_inst, ccode_days, ccode_days/total_contact_days as ccode_ratio, (surg_days + rad_days + chemproc_days + chemdrug_inst)/total_contact_days as treatment_ratio
from full_patient_list_filtered_rad_surg_chemproc_drug_dx 
where avg_days_bt_visits is not null